In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

builder = SparkSession.builder \
    .appName("PySparkDemo") \
    .master("local[*]") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    ) \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession created successfully.")

In [ ]:
from pathlib import Path
base_path=Path("").resolve()

input_path = base_path / "Input"

csv_path = input_path / "retail_sales.csv"
print(csv_path)

In [ ]:
raw_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv(str(csv_path))

raw_df.show(truncate=False)

raw_df.printSchema()

In [ ]:
bronze_df = raw_df.withColumn(
    "IngestionTimestamp",
    F.current_timestamp()
).withColumn(
    "SourceFile",
    F.input_file_name()
).withColumn(
    "SourceSystem",
    F.lit("Retail POS")
)

bronze_df.show(truncate=False)

In [ ]:
bronze_path = str(
    base_path / "bronze" / "retail_sales"
)

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

In [ ]:
bronze_sales_df = spark.read \
    .format("delta") \
    .load(bronze_path)

bronze_sales_df.show(truncate=False)

print(
    "Bronze Record Count:",
    bronze_sales_df.count()
)

In [ ]:
typed_df = bronze_sales_df \
    .withColumn(
        "SalesID",
        F.expr("try_cast(SalesID AS INT)")
    ) \
    .withColumn(
        "CustomerID",
        F.expr("try_cast(CustomerID AS INT)")
    ) \
    .withColumn(
        "Quantity",
        F.expr("try_cast(Quantity AS INT)")
    ) \
    .withColumn(
        "Price",
        F.expr("try_cast(Price AS DECIMAL(12,2))")
    ) \
    .withColumn(
        "SalesDate",
        F.expr("try_cast(SalesDate AS DATE)")
    )

typed_df.printSchema()

In [ ]:
standardized_df = typed_df \
    .withColumn(
        "ProductName",
        F.initcap(F.trim("ProductName"))
    ) \
    .withColumn(
        "Category",
        F.initcap(F.trim("Category"))
    ) \
    .withColumn(
        "City",
        F.initcap(F.trim("City"))
    )

standardized_df.show(truncate=False)

In [ ]:
validated_df = standardized_df.withColumn(
    "RejectionReason",

    F.when(
        F.col("SalesID").isNull(),
        "Invalid SalesID"
    ).when(
        F.col("CustomerID").isNull(),
        "Invalid CustomerID"
    ).when(
        F.col("ProductName").isNull() |
        (F.col("ProductName") == ""),
        "Missing ProductName"
    ).when(
        F.col("Category").isNull() |
        (F.col("Category") == ""),
        "Missing Category"
    ).when(
        F.col("Quantity").isNull() |
        (F.col("Quantity") <= 0),
        "Invalid Quantity"
    ).when(
        F.col("Price").isNull() |
        (F.col("Price") <= 0),
        "Invalid Price"
    ).when(
        F.col("SalesDate").isNull(),
        "Invalid SalesDate"
    ).otherwise(
        F.lit(None).cast("string")
    )
)

In [ ]:
rejected_df = validated_df.filter(
    F.col("RejectionReason").isNotNull()
)

rejected_df.select(
    "SalesID",
    "ProductName",
    "Quantity",
    "Price",
    "SalesDate",
    "RejectionReason"
).show(truncate=False)

In [ ]:
rejected_path = str(
    base_path / "silver" / "rejected_records"
)

rejected_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(rejected_path)

In [ ]:
valid_df = validated_df.filter(
    F.col("RejectionReason").isNull()
).drop("RejectionReason")

deduplicated_df = valid_df.dropDuplicates(
    ["SalesID"]
)

print("Before Deduplication:", valid_df.count())

print(
    "After Deduplication:",
    deduplicated_df.count()
)

In [ ]:
silver_df = deduplicated_df \
    .withColumn(
        "TotalAmount",
        F.col("Quantity") * F.col("Price")
    ) \
    .withColumn(
        "SalesYear",
        F.year("SalesDate")
    ) \
    .withColumn(
        "SalesMonth",
        F.month("SalesDate")
    ) \
    .withColumn(
        "MonthName",
        F.date_format("SalesDate", "MMMM")
    ) \
    .withColumn(
        "SalesQuarter",
        F.quarter("SalesDate")
    ) \
    .withColumn(
        "ProcessedTimestamp",
        F.current_timestamp()
    )

silver_df.show(truncate=False)

In [ ]:
silver_path = str(
    base_path / "silver" / "retail_sales"
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [ ]:
silver_sales_df = spark.read \
    .format("delta") \
    .load(silver_path)

silver_sales_df.select(
    "SalesID",
    "ProductName",
    "Category",
    "Quantity",
    "Price",
    "TotalAmount"
).orderBy("SalesID").show(truncate=False)

print(
    "Silver Record Count:",
    silver_sales_df.count()
)

In [ ]:
category_sales_df = silver_sales_df \
    .groupBy("Category") \
    .agg(
        F.count("*").alias("TotalTransactions"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.sum("TotalAmount").alias("TotalRevenue"),
        F.avg("TotalAmount").alias("AverageSale")
    ) \
    .orderBy(F.desc("TotalRevenue"))

category_sales_df.show(truncate=False)

In [ ]:
category_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(str(
        base_path / "gold" / "category_sales"
    ))

In [ ]:
monthly_sales_df = silver_sales_df \
    .groupBy(
        "SalesYear",
        "SalesMonth",
        "MonthName"
    ) \
    .agg(
        F.count("*").alias("TotalTransactions"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.sum("TotalAmount").alias("TotalRevenue")
    ) \
    .orderBy("SalesYear", "SalesMonth")

monthly_sales_df.show(truncate=False)

In [ ]:
monthly_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(str(
        base_path / "gold" / "monthly_sales"
    ))

In [ ]:
customer_sales_df = silver_sales_df \
    .groupBy("CustomerID") \
    .agg(
        F.count("*").alias("TotalOrders"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.sum("TotalAmount").alias("TotalSpent")
    ) \
    .withColumn(
        "CustomerSegment",
        F.when(
            F.col("TotalSpent") >= 100000,
            "Premium"
        ).when(
            F.col("TotalSpent") >= 50000,
            "Gold"
        ).otherwise("Regular")
    ) \
    .orderBy(F.desc("TotalSpent"))

customer_sales_df.show(truncate=False)

In [ ]:
customer_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(str(
        base_path / "gold" / "customer_sales"
    ))

In [ ]:
city_sales_df = silver_sales_df \
    .groupBy("City") \
    .agg(
        F.count("*").alias("TotalOrders"),
        F.countDistinct(
            "CustomerID"
        ).alias("UniqueCustomers"),
        F.sum("TotalAmount").alias("TotalRevenue")
    ) \
    .orderBy(F.desc("TotalRevenue"))

city_sales_df.show(truncate=False)

In [ ]:
city_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(str(
        base_path / "gold" / "city_sales"
    ))

In [ ]:
product_sales_df = silver_sales_df \
    .groupBy(
        "ProductName",
        "Category"
    ) \
    .agg(
        F.sum("Quantity").alias("UnitsSold"),
        F.sum("TotalAmount").alias("TotalRevenue")
    ) \
    .orderBy(F.desc("TotalRevenue"))

product_sales_df.show(truncate=False)

In [ ]:
product_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(str(
        base_path / "gold" / "product_sales"
    ))

In [ ]:
silver_sales_df.filter(
    F.col("Category") == "Electronics"
).select(
    "ProductName",
    "TotalAmount"
).show()

In [ ]:
silver_sales_df = spark.read \
    .format("delta") \
    .load(silver_path)

silver_sales_df.createOrReplaceTempView(
    "RetailSales"
)

In [ ]:
spark.sql("""
    SELECT *
    FROM RetailSales Order By Price DESC
""").show(truncate=False)

In [ ]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW ElectronicsSales AS
    SELECT *
    FROM RetailSales
    WHERE Category = 'Electronics'
""")

In [ ]:
spark.sql("""
    SELECT
        ProductName,
        Quantity,
        TotalAmount
    FROM ElectronicsSales
""").show()

In [ ]:
silver_sales_df.createOrReplaceGlobalTempView(
    "GlobalRetailSales"
)


In [ ]:
spark.sql("""
    SELECT *
    FROM global_temp.GlobalRetailSales
""").show()

In [ ]:
new_spark = spark.newSession()

new_spark.sql("""
    SELECT *
    FROM RetailSales
""").show()

In [ ]:
spark.catalog.listTables()

In [ ]:
spark.sql("SHOW TABLES").show()

spark.sql(
    "SHOW TABLES IN global_temp"
).show()

In [ ]:
spark.catalog.dropTempView(
    "ElectronicsSales"
)

In [49]:
spark.catalog.dropGlobalTempView(
    "GlobalRetailSales"
)

True